# RoBERTa-base Organizer-Style Baseline (PyTorch only)

Implements organizer baseline choices for Task 1 using a Hugging Face PyTorch `Trainer` pipeline.

Key organizer-aligned choices:
- Binary PCL setup (`0/1 -> 0`, `2/3/4 -> 1`)
- Downsample negatives in train to a 1:2 positive:negative ratio
- Train for exactly 1 epoch

No TensorFlow modules are used.

In [1]:
import random
from pathlib import Path

import numpy as np
import pandas as pd
import torch
from sklearn.metrics import classification_report, precision_recall_fscore_support
from sklearn.model_selection import train_test_split

from transformers import (
    AutoModelForSequenceClassification,
    AutoTokenizer,
    DataCollatorWithPadding,
    Trainer,
    TrainingArguments,
    set_seed,
)

SEED = 42
MODEL_NAME = "roberta-base"
MAX_LENGTH = 256
BATCH_SIZE = 16
EVAL_BATCH_SIZE = 32
LEARNING_RATE = 2e-5
WEIGHT_DECAY = 0.01
EPOCHS = 1

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)
set_seed(SEED)

cwd = Path.cwd().resolve()
if (cwd / "data").exists():
    PROJECT_ROOT = cwd
elif (cwd.parent / "data").exists():
    PROJECT_ROOT = cwd.parent
else:
    raise FileNotFoundError("Could not locate project root containing data/ directory.")

RAW_DATA_DIR = PROJECT_ROOT / "data" / "raw"
MODEL_DIR = PROJECT_ROOT / "models" / "roberta_organizer_style_pytorch"
OUTPUT_DIR = PROJECT_ROOT / "output"

MODEL_DIR.mkdir(parents=True, exist_ok=True)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")
print(f"Project root: {PROJECT_ROOT}")

/vol/bitbucket/tq422/nlpenv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Using device: cuda
Project root: /vol/bitbucket/tq422/NLP_cw


## Load data, split train/dev, and apply organizer-style downsampling

In [2]:
df = pd.read_csv(
    RAW_DATA_DIR / "dontpatronizeme_pcl.tsv",
    sep="\t",
    skiprows=4,
    names=["par_id", "art_id", "keyword", "country_code", "text", "label"],
)

# Organizer mapping for Task 1 binary labels.
df["label_binary"] = df["label"].apply(lambda x: 0 if x in [0, 1] else 1)

a_train = pd.read_csv(RAW_DATA_DIR / "train_semeval_parids-labels.csv")
a_dev = pd.read_csv(RAW_DATA_DIR / "dev_semeval_parids-labels.csv")

train_ids = set(a_train["par_id"])
dev_ids = set(a_dev["par_id"])

train_df = df[df["par_id"].isin(train_ids)].copy()
dev_df = df[df["par_id"].isin(dev_ids)].copy()

train_df = train_df[["par_id", "text", "label_binary"]].dropna(subset=["text"])
dev_df = dev_df[["par_id", "text", "label_binary"]]

train_df["text"] = train_df["text"].astype(str)
dev_df["text"] = dev_df["text"].astype(str)

# Organizer baseline choice: keep all positives + first 2x negatives.
pcl_df = train_df[train_df["label_binary"] == 1]
npos = len(pcl_df)
train_df_balanced = pd.concat(
    [pcl_df, train_df[train_df["label_binary"] == 0].iloc[: npos * 2]],
    ignore_index=True,
)

# Split a validation set from organizer-style train; dev is held out and not used during training
VAL_RATIO = 0.1
train_df_balanced, val_df = train_test_split(
    train_df_balanced,
    test_size=VAL_RATIO,
    stratify=train_df_balanced["label_binary"],
    random_state=SEED,
)
train_df_balanced = train_df_balanced.reset_index(drop=True)
val_df = val_df.reset_index(drop=True)

print(f"Full train size: {len(train_df)}")
print(f"Organizer-style train size: {len(train_df_balanced)}")
print(f"Val size: {len(val_df)}")
print(f"Dev size: {len(dev_df)} (held out, not used during training)")
print("\nOrganizer-style train label distribution:")
print(train_df_balanced["label_binary"].value_counts())
print("\nVal label distribution:")
print(val_df["label_binary"].value_counts())
print("\nDev label distribution:")
print(dev_df["label_binary"].value_counts())

Full train size: 8375
Organizer-style train size: 2143
Val size: 239
Dev size: 2094 (held out, not used during training)

Organizer-style train label distribution:
label_binary
0    1429
1     714
Name: count, dtype: int64

Val label distribution:
label_binary
0    159
1     80
Name: count, dtype: int64

Dev label distribution:
label_binary
0    1895
1     199
Name: count, dtype: int64


In [3]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

class PCLDataset(torch.utils.data.Dataset):
    def __init__(self, texts, labels, tokenizer, max_length=256):
        self.texts = list(texts)
        self.labels = list(labels)
        self.tokenizer = tokenizer
        self.max_length = max_length

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        enc = self.tokenizer(
            self.texts[idx],
            truncation=True,
            max_length=self.max_length,
            return_attention_mask=True,
        )
        enc["labels"] = int(self.labels[idx])
        return enc

train_dataset = PCLDataset(
    train_df_balanced["text"],
    train_df_balanced["label_binary"],
    tokenizer,
    max_length=MAX_LENGTH,
)
val_dataset = PCLDataset(val_df["text"], val_df["label_binary"], tokenizer, max_length=MAX_LENGTH)
dev_dataset = PCLDataset(dev_df["text"], dev_df["label_binary"], tokenizer, max_length=MAX_LENGTH)
data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

print("Tokenizer and datasets ready.")

Tokenizer and datasets ready.


## Train (1 epoch, organizer-style)

In [4]:
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)

    p_pos, r_pos, f1_pos, _ = precision_recall_fscore_support(
        labels, preds, labels=[1], average=None, zero_division=0
    )
    _, _, f1_macro, _ = precision_recall_fscore_support(
        labels, preds, average="macro", zero_division=0
    )

    acc = (preds == labels).mean()

    return {
        "accuracy": float(acc),
        "precision_pos": float(p_pos[0]),
        "recall_pos": float(r_pos[0]),
        "f1_pos": float(f1_pos[0]),
        "f1_macro": float(f1_macro),
    }

model = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME, num_labels=2)

training_args = TrainingArguments(
    output_dir=str(MODEL_DIR / "checkpoints"),
    learning_rate=LEARNING_RATE,
    per_device_train_batch_size=BATCH_SIZE,
    per_device_eval_batch_size=EVAL_BATCH_SIZE,
    num_train_epochs=EPOCHS,
    weight_decay=WEIGHT_DECAY,
    warmup_ratio=0.1,
    eval_strategy="epoch",
    save_strategy="epoch",
    logging_strategy="steps",
    logging_steps=50,
    load_best_model_at_end=True,
    metric_for_best_model="f1_pos",
    greater_is_better=True,
    report_to="none",
    seed=SEED,
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    processing_class=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
)

print("Trainer initialized.")

Some weights of RobertaForSequenceClassification were not initialized from the model checkpoint at roberta-base and are newly initialized: ['classifier.dense.bias', 'classifier.dense.weight', 'classifier.out_proj.bias', 'classifier.out_proj.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Trainer initialized.


In [5]:
train_result = trainer.train()
print("Training finished.")

Epoch,Training Loss,Validation Loss,Accuracy,Precision Pos,Recall Pos,F1 Pos,F1 Macro
1,0.511600,0.402720,0.832636,0.794118,0.675000,0.729730,0.804259


Training finished.


In [6]:
eval_metrics = trainer.evaluate(dev_dataset)
print("Dev metrics:")
for k, v in eval_metrics.items():
    if isinstance(v, (int, float)):
        print(f"{k}: {v:.4f}")
    else:
        print(f"{k}: {v}")

pred_output = trainer.predict(dev_dataset)
dev_preds = np.argmax(pred_output.predictions, axis=-1)
dev_labels = pred_output.label_ids

import json
report_str = classification_report(dev_labels, dev_preds, target_names=["No PCL", "PCL"], digits=4, zero_division=0)
print("\nClassification report (dev):")
print(report_str)


with open(PROJECT_ROOT / "baseline_dev.txt", "w", encoding="utf-8") as f:
    for pr in dev_preds:
        f.write(f"{pr}\n")
print(f"Saved dev predictions to {PROJECT_ROOT / 'dev.txt'} (root)")

Dev metrics:
eval_loss: 0.2973
eval_accuracy: 0.8586
eval_precision_pos: 0.3700
eval_recall_pos: 0.6935
eval_f1_pos: 0.4825
eval_f1_macro: 0.7003
eval_runtime: 6.8904
eval_samples_per_second: 303.9020
eval_steps_per_second: 9.5790
epoch: 1.0000

Classification report (dev):
              precision    recall  f1-score   support

      No PCL     0.9646    0.8760    0.9181      1895
         PCL     0.3700    0.6935    0.4825       199

    accuracy                         0.8586      2094
   macro avg     0.6673    0.7847    0.7003      2094
weighted avg     0.9081    0.8586    0.8767      2094

Saved dev predictions to /vol/bitbucket/tq422/NLP_cw/dev.txt (root)


In [ ]:
# trainer.save_model(str(MODEL_DIR))
# tokenizer.save_pretrained(str(MODEL_DIR))
# print(f"Saved model and tokenizer to: {MODEL_DIR}")